# 06: Multistream Pipeline

This notebook extends the work done in Notebook 01 and Notebook 02. It focuses on building a multistream data pipeline that can handle multiple synchronized streams from the VRS recording.

This next notebook will generalize that approach from one stream to all relevant streams.

## In this notebook, we will:

1. Inventory all streams (Reuse the discovery logic from Notebook 01)
- Build a canonical table with stream id, label, modality, sample count, and time coverage.
- Separate image, IMU, and auxiliary streams clearly.

2. Validate per-stream access
- Confirm that each stream can be decoded or read by index.
- Measure timestamp monotonicity and sample spacing per stream.
- Flag empty streams, sparse streams, and streams with non-standard timing.

3. Define a multistream dataset abstraction
- Extend the single-stream dataset pattern from Notebook 02.
- Decide whether one item should return a dictionary of synchronized streams or a per-stream record bundle.
- Keep the API compatible with PyTorch `Dataset` and `DataLoader`.

4. Build synchronization and sampling rules
- Align streams on a shared time axis or a reference stream.
- Define how to handle missing samples, different rates, and partial overlap.
- Add deterministic indexing rules for training and analysis use cases.

5. Add visualization and sanity checks
- Display frame-aligned samples across multiple streams.
- Inspect synchronization quality on a few representative windows.
- Verify that the outputs are physically consistent before exporting code.

6. Export final artifacts
- Save stream tables, quality reports, and any derived indices under `data/processed/`.
- Document the chosen multistream data model and any assumptions about synchronization.

7. Package reusable code into `scripts/`
- Move stable helpers into Python modules following package conventions.
- Separate stream discovery, indexing, dataset definitions, and transforms into dedicated files.
- Keep notebooks focused on experimentation and validation, not on long-term implementation.

As a sample, we will use `kettle_and_forklift_recording.vrs` and its associated `kettle_and_forklift_recording.json` metadata, located in `data/raw/kettle_and_forklift`. The goal is to create a robust multistream pipeline that can be easily adapted to other recordings.

## 6.1 Setup and Recording Discovery

This section prepares the notebook for multistream work.
We reuse the same style established in Notebook 01 and Notebook 02:
- validate the recording paths
- make the project modules importable
- open the VRS provider
- discover the available streams as the first canonical inventory step


In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from projectaria_tools.core import data_provider, sensor_data


In [2]:
# Define the sample recording used for the multistream pipeline.
DATA_DIR = Path("..") / "data" / "raw" / "kettle_and_forklift"
VRS_PATH = DATA_DIR / "kettle_and_forklift_recording.vrs"
JSON_PATH = DATA_DIR / "kettle_and_forklift_recording.json"
OUT_DIR = Path("..") / "data" / "processed" / "kettle_and_forklift"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Validate the required input files before continuing.
if not VRS_PATH.exists():
    raise FileNotFoundError(f"VRS file not found: {VRS_PATH.resolve()}")
if not JSON_PATH.exists():
    raise FileNotFoundError(f"JSON metadata file not found: {JSON_PATH.resolve()}")

print("VRS path:", VRS_PATH.resolve())
print("JSON path:", JSON_PATH.resolve())
print("Output dir:", OUT_DIR.resolve())


VRS path: C:\Users\Gmalv\Desktop\Scuola_Universita\AriaGen1\data\raw\kettle_and_forklift\kettle_and_forklift_recording.vrs
JSON path: C:\Users\Gmalv\Desktop\Scuola_Universita\AriaGen1\data\raw\kettle_and_forklift\kettle_and_forklift_recording.json
Output dir: C:\Users\Gmalv\Desktop\Scuola_Universita\AriaGen1\data\processed\kettle_and_forklift


In [3]:
# Make the repository root importable so helper modules in scripts/ can be reused.
project_root = next((p for p in [Path.cwd().resolve(), Path.cwd().resolve().parent] if (p / "scripts" / "aria_dataset.py").exists()), None)
if project_root is None:
    raise RuntimeError("Could not find the project root containing scripts/aria_dataset.py")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Reuse the frame extraction helper established in Notebook 02.
from scripts.aria_dataset import _extract_image_array_from_sample

# Create the VRS provider used by the rest of the notebook.
provider = data_provider.create_vrs_data_provider(str(VRS_PATH))
if provider is None:
    raise RuntimeError("Failed to initialize the VRS data provider.")

print("Provider initialized successfully.")
print("Available streams:", len(provider.get_all_streams()))

Provider initialized successfully.
Available streams: 11


## 6.2 Stream Inventory and Canonical Table

This section recreates the stream discovery pattern from Notebook 01, but with a multistream perspective.
We want one canonical table that makes the recording structure explicit before any dataset or synchronization logic is introduced.

The table will be the base artifact for the rest of the notebook, every later decision should refer back to this.

In [4]:
# Helper to classify streams into a broad modality category.
def infer_modality(label: str) -> str:
    """Infer a broad modality label from the stream name or label."""
    text = (label or "").lower()
    if any(key in text for key in ["camera", "image", "rgb", "slam", "et"]):
        return "image"
    if any(key in text for key in ["imu", "accel", "gyro"]):
        return "imu"
    if any(key in text for key in ["audio", "mic"]):
        return "audio"
    if any(key in text for key in ["gps", "baro", "mag", "bluetooth", "ble", "wifi", "wps"]):
        return "auxiliary"
    return "unknown"


In [9]:
# Build a canonical inventory table for all available streams.
stream_rows = []
for stream_id in provider.get_all_streams():
    stream_label = provider.get_label_from_stream_id(stream_id)
    sample_count = provider.get_num_data(stream_id)
    stream_rows.append({
        "stream_id": str(stream_id),
        "label": str(stream_label),
        "type": infer_modality(str(stream_label)),
        "sample_count": int(sample_count),
    })

streams_df = pd.DataFrame(stream_rows).sort_values(["type", "label", "stream_id"]).reset_index(drop=True)

print(f"Discovered {len(streams_df)} streams.")
display(streams_df)

Discovered 11 streams.


,stream_id,label,type,sample_count
0,247-1,baro0,auxiliary,2565
1,281-1,gps,auxiliary,52
2,1203-1,mag0,auxiliary,516
3,282-1,wps,auxiliary,136
4,283-1,bluetooth,image,0
5,211-1,camera-et,image,1551
6,214-1,camera-rgb,image,1551
7,1201-1,camera-slam-left,image,1551
8,1201-2,camera-slam-right,image,1551
9,1202-2,imu-left,imu,41675


In [10]:
# Keep a short summary that can be reused by later sections.
stream_summary_df = streams_df.groupby("type", dropna=False).size().reset_index(name="num_streams")
display(stream_summary_df)

,type,num_streams
0,auxiliary,4
1,image,5
2,imu,2


## 6.3 MultiStream Timestamp Validation

The next step is to validate access and timing for each stream in a reusable way.


In [11]:
# Helper to resolve a stream identifier back to the provider stream object.
def resolve_stream_id(provider_obj, stream_id_str: str):
    """Resolve a string stream id back to the provider-native stream object."""
    matches = [sid for sid in provider_obj.get_all_streams() if str(sid) == str(stream_id_str)]
    if len(matches) == 0:
        return None
    return matches[0]


# Helper to read timestamps for one stream using index-based access.
def get_stream_timestamps_ns(provider_obj, stream_id_obj):
    """Read all DEVICE_TIME timestamps for a stream as a NumPy int64 array."""
    sample_count = int(provider_obj.get_num_data(stream_id_obj))
    timestamps_ns = np.empty(sample_count, dtype=np.int64)
    for index in range(sample_count):
        sample = provider_obj.get_sensor_data_by_index(stream_id_obj, index)
        timestamps_ns[index] = sample.get_time_ns(sensor_data.TimeDomain.DEVICE_TIME)
    return timestamps_ns


# Helper to compute a compact timing summary from a timestamp array.
def summarize_timestamp_series(timestamps_ns):
    """Compute timing statistics for one timestamp series."""
    timestamps_ns = np.asarray(timestamps_ns, dtype=np.int64)
    if timestamps_ns.size == 0:
        return {
            "sample_count": 0,
            "t_min_ns": None,
            "t_max_ns": None,
            "duration_s": 0.0,
            "delta_min_ms": None,
            "delta_median_ms": None,
            "delta_max_ms": None,
            "strictly_increasing": True,
        }

    deltas_ns = np.diff(timestamps_ns)
    positive_deltas_ns = deltas_ns[deltas_ns > 0]
    return {
        "sample_count": int(timestamps_ns.size),
        "t_min_ns": int(timestamps_ns.min()),
        "t_max_ns": int(timestamps_ns.max()),
        "duration_s": float((timestamps_ns.max() - timestamps_ns.min()) / 1e9) if timestamps_ns.size > 1 else 0.0,
        "delta_min_ms": float(positive_deltas_ns.min() / 1e6) if positive_deltas_ns.size > 0 else None,
        "delta_median_ms": float(np.median(positive_deltas_ns) / 1e6) if positive_deltas_ns.size > 0 else None,
        "delta_max_ms": float(positive_deltas_ns.max() / 1e6) if positive_deltas_ns.size > 0 else None,
        "strictly_increasing": bool(np.all(deltas_ns > 0)) if deltas_ns.size > 0 else True,
    }


# Build the per-stream timing report for the full inventory.
timestamp_rows = []
for _, stream_row in streams_df.iterrows():
    stream_id_obj = resolve_stream_id(provider, stream_row["stream_id"])
    if stream_id_obj is None:
        timestamp_rows.append({
            "stream_id": stream_row["stream_id"],
            "label": stream_row["label"],
            "type": stream_row["type"],
            "sample_count": 0,
            "t_min_ns": None,
            "t_max_ns": None,
            "duration_s": None,
            "delta_min_ms": None,
            "delta_median_ms": None,
            "delta_max_ms": None,
            "strictly_increasing": False,
            "status": "UNRESOLVABLE",
        })
        continue

    try:
        timestamps_ns = get_stream_timestamps_ns(provider, stream_id_obj)
        timing_summary = summarize_timestamp_series(timestamps_ns)
        timing_summary.update({
            "stream_id": stream_row["stream_id"],
            "label": stream_row["label"],
            "type": stream_row["type"],
            "status": "OK",
        })
        timestamp_rows.append(timing_summary)
    except Exception as exc:
        timestamp_rows.append({
            "stream_id": stream_row["stream_id"],
            "label": stream_row["label"],
            "type": stream_row["type"],
            "sample_count": int(stream_row["sample_count"]),
            "t_min_ns": None,
            "t_max_ns": None,
            "duration_s": None,
            "delta_min_ms": None,
            "delta_median_ms": None,
            "delta_max_ms": None,
            "strictly_increasing": False,
            "status": f"ERROR: {exc}",
        })

# Build and format the DataFrame with a stable, explicit column order.
stream_timing_df = pd.DataFrame(timestamp_rows)
stream_timing_df = stream_timing_df.sort_values(["status", "type", "label", "stream_id"]).reset_index(drop=True)

ordered_columns = [
    "stream_id",
    "label",
    "type",
    "sample_count",
    "t_min_ns",
    "t_max_ns",
    "duration_s",
    "delta_min_ms",
    "delta_median_ms",
    "delta_max_ms",
    "strictly_increasing",
    "status",
]
stream_timing_df = stream_timing_df[ordered_columns]

display(stream_timing_df)

,stream_id,label,type,sample_count,t_min_ns,t_max_ns,duration_s,delta_min_ms,delta_median_ms,delta_max_ms,strictly_increasing,status
0,247-1,baro0,auxiliary,2565,8.883402e+11,9.400196e+11,51.679389,19.325863,20.159750,20.978412,True,OK
1,281-1,gps,auxiliary,52,8.886455e+11,9.396473e+11,51.001845,994.000677,999.652656,1005.927447,True,OK
2,1203-1,mag0,auxiliary,516,8.884287e+11,9.399679e+11,51.539196,100.007125,100.084300,100.161875,True,OK
3,282-1,wps,auxiliary,136,-1.000000e+00,-1.000000e+00,0.000000,NaN,NaN,NaN,False,OK
4,283-1,bluetooth,image,0,NaN,NaN,0.000000,NaN,NaN,NaN,True,OK
5,211-1,camera-et,image,1551,8.883627e+11,9.400211e+11,51.658400,33.328000,33.328000,33.328000,True,OK
6,214-1,camera-rgb,image,1551,8.883626e+11,9.400210e+11,51.658395,33.037587,33.327338,33.614500,True,OK
7,1201-1,camera-slam-left,image,1551,8.883627e+11,9.400211e+11,51.658400,33.328000,33.328000,33.328000,True,OK
8,1201-2,camera-slam-right,image,1551,8.883627e+11,9.400211e+11,51.658400,33.328000,33.328000,33.328000,True,OK
9,1202-2,imu-left,imu,41675,8.883457e+11,9.400311e+11,51.685367,1.240050,1.240238,1.240413,True,OK


### Interpretation

This table is the first quality gate for the multistream pipeline.
It tells us which streams are valid for downstream use, which streams are empty or sparse, and whether the timestamps behave as expected.

Once this report is stable, we can safely move on to stream-level integrity checks and dataset design.

### Note on sparse auxiliary streams

It is normal for `wps` and `bluetooth` to show many `NaN` values in later timing or quality tables.
Especially `bluetooth` was turned off during the recording, so it is expected to be empty.
For that reason, they should be treated as auxiliary streams and analyzed separately from the main synchronized recording streams.

## 6.4 Sample Read Integrity Checks

This section checks a few representative indices per stream.
The goal is not to exhaustively test every sample, but to confirm that index-based access works consistently across the inventory.

### 6.4.1 Integrity Check Strategy

We use a lightweight probing strategy to validate per-stream readability:
- resolve stream id from the canonical table
- test a small set of representative indices
- report pass/fail counts and a compact status

This gives a robust sanity check without scanning every sample.

In [ ]:
# Helper to select representative sample indices for quick integrity probing.
def select_probe_indices(sample_count: int):
    """Select a compact set of representative indices for one stream."""
    if sample_count <= 0:
        return []

    candidate_indices = [
        0,
        sample_count // 4,
        sample_count // 2,
        (3 * sample_count) // 4,
        sample_count - 1,
    ]

    # Remove duplicates while preserving order.
    probe_indices = []
    seen = set()
    for idx in candidate_indices:
        idx = int(idx)
        if idx not in seen:
            probe_indices.append(idx)
            seen.add(idx)
    return probe_indices


# Helper to probe index-based readability for one stream.
def probe_stream_readability(provider_obj, stream_id_obj, probe_indices):
    """Attempt to read a list of indices and return integrity counters."""
    read_ok = 0
    read_fail = 0
    first_error = None

    for idx in probe_indices:
        try:
            _ = provider_obj.get_sensor_data_by_index(stream_id_obj, int(idx))
            read_ok += 1
        except Exception as exc:
            read_fail += 1
            if first_error is None:
                first_error = str(exc)

    if read_fail == 0:
        status = "OK"
    elif read_ok > 0:
        status = "PARTIAL"
    else:
        status = "ERROR"

    return {
        "tested_indices": probe_indices,
        "read_ok": read_ok,
        "read_fail": read_fail,
        "status": status,
        "error_example": first_error,
    }

In [ ]:
# Build the stream integrity report using the modular probe helpers.
integrity_rows = []
for _, stream_row in streams_df.iterrows():
    stream_id_str = stream_row["stream_id"]
    stream_id_obj = resolve_stream_id(provider, stream_id_str)
    sample_count = int(stream_row["sample_count"])

    # Handle unresolved stream ids explicitly.
    if stream_id_obj is None:
        integrity_rows.append({
            "stream_id": stream_id_str,
            "label": stream_row["label"],
            "type": stream_row["type"],
            "sample_count": sample_count,
            "tested_indices": [],
            "read_ok": 0,
            "read_fail": 0,
            "status": "UNRESOLVABLE",
            "error_example": "Stream id not resolvable from provider.",
        })
        continue

    # Handle empty streams as a valid special case.
    if sample_count == 0:
        integrity_rows.append({
            "stream_id": stream_id_str,
            "label": stream_row["label"],
            "type": stream_row["type"],
            "sample_count": 0,
            "tested_indices": [],
            "read_ok": 0,
            "read_fail": 0,
            "status": "EMPTY",
            "error_example": None,
        })
        continue

    probe_indices = select_probe_indices(sample_count)
    probe_result = probe_stream_readability(provider, stream_id_obj, probe_indices)

    integrity_rows.append({
        "stream_id": stream_id_str,
        "label": stream_row["label"],
        "type": stream_row["type"],
        "sample_count": sample_count,
        "tested_indices": probe_result["tested_indices"],
        "read_ok": probe_result["read_ok"],
        "read_fail": probe_result["read_fail"],
        "status": probe_result["status"],
        "error_example": probe_result["error_example"],
    })

stream_integrity_df = pd.DataFrame(integrity_rows)
stream_integrity_df = stream_integrity_df.sort_values(["status", "type", "label", "stream_id"]).reset_index(drop=True)

display(stream_integrity_df)

# Also expose a compact status summary for quick inspection.
integrity_status_df = stream_integrity_df.groupby("status", dropna=False).size().reset_index(name="num_streams")
display(integrity_status_df)

### 6.4.2 Integrity Results Interpretation

Interpretation guidelines:
- `OK`: all probed indices were read successfully.
- `PARTIAL`: only a subset of probed indices was readable.
- `ERROR`: all probes failed for that stream.
- `EMPTY`: the stream has no samples in this recording.
- `UNRESOLVABLE`: stream id could not be mapped back to a provider stream object.

This report complements the timestamp validation from Section 6.3 and forms the second quality gate before synchronization logic.